In [5]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

# ============================================================
# 1. PARÁMETROS Y UNIVERSO
# ============================================================

START_DATE = "2010-01-01"
END_DATE = "2024-12-30"
WINDOW = 60
GAMMA = 1.0
TOL = 1e-8
N = 50

universe = pd.read_csv("universe_50_2009.csv")

tickers = universe["ticker_yf"].dropna().unique()
display(tickers)

print(f"Número de activos del universo: {len(tickers)}")

# ============================================================
# 2. PRECIOS DIARIOS Y RENTABILIDADES MENSUALES
# ============================================================

data = yf.download(
    tickers=list(tickers),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=True,
    threads=True
)

# Último precio disponible de cada mes.
monthly_prices = data["Close"].resample("ME").last()

# fill_method=None evita rellenar artificialmente precios ausentes.
returns = monthly_prices.pct_change(fill_method=None).iloc[1:].copy()

print(f"Observaciones mensuales: {returns.shape[0]}")
print(f"Primera rentabilidad: {returns.index.min():%Y-%m-%d}")
print(f"Última rentabilidad: {returns.index.max():%Y-%m-%d}")

missing_by_asset = returns.isna().sum()

if missing_by_asset.sum() > 0:
    print("\nValores ausentes por activo:")
    print(missing_by_asset[missing_by_asset > 0])

    raise ValueError(
        "Existen rentabilidades ausentes. Antes del backtest debes decidir "
        "cómo tratar los activos incompletos; no continúes con una matriz parcial."
    )

assert returns.shape[0] > WINDOW, "No hay suficientes meses para el backtest."
assert returns.shape[1] == N
assert returns.index.is_monotonic_increasing
assert not returns.index.duplicated().any()

# ============================================================
# 3. FUNCIONES Y BACKTEST UNIFICADO
# ============================================================

# ------------------------------------------------------------
# Estimadores de covarianza
# ------------------------------------------------------------

def sample_covariance(returns_window):
    """
    Covarianza muestral clásica.
    """
    cov = returns_window.cov().to_numpy()

    return cov, np.nan


def ledoit_wolf_covariance(returns_window):
    """
    Covarianza Ledoit-Wolf e intensidad de shrinkage.
    """
    lw = LedoitWolf(assume_centered=False)
    lw.fit(returns_window.to_numpy())

    return lw.covariance_, lw.shrinkage_


# ------------------------------------------------------------
# Funciones auxiliares
# ------------------------------------------------------------

def linear_solve(cov, vector):
    """
    Resuelve cov @ x = vector sin calcular la inversa explícita.
    """
    try:
        return np.linalg.solve(cov, vector)

    except np.linalg.LinAlgError:
        warnings.warn(
            "Matriz singular: se utiliza pseudoinversa.",
            RuntimeWarning
        )
        return np.linalg.pinv(cov) @ vector


def solve_markowitz(
    mu,
    cov,
    gamma=GAMMA,
    long_only=False,
    tol=TOL
):
    """
    Resuelve numéricamente:

        max w' mu - (gamma / 2) w' Sigma w
        s.a. 1' w = 1

    Si long_only=True, impone w_i >= 0.
    """
    N = len(mu)
    initial_weights = np.repeat(1/N, N)

    def objective(w):
        return -(w @ mu - gamma / 2 * w @ cov @ w)

    def gradient(w):
        return -(mu - gamma * cov @ w)

    bounds = (
        [(0.0, None)] * N
        if long_only
        else [(None, None)] * N
    )

    result = minimize(
        objective,
        initial_weights,
        jac=gradient,
        method="SLSQP",
        bounds=bounds,
        constraints={
            "type": "eq",
            "fun": lambda w: w.sum() - 1.0,
            "jac": lambda w: np.ones(N)
        },
        options={
            "ftol": tol,
            "maxiter": 1000,
            "disp": False
        }
    )

    if not result.success:
        raise RuntimeError(
            f"Optimización Markowitz fallida: {result.message}"
        )

    return result.x


def solve_minimum_variance(
    cov,
    long_only=True,
    tol=TOL
):
    """
    Resuelve:

        min w' Sigma w
        s.a. 1' w = 1

    Por defecto se usa long-only para que sea un benchmark invertible.
    """
    N = cov.shape[0]
    initial_weights = np.repeat(1/N, N)

    def objective(w):
        return w @ cov @ w

    def gradient(w):
        return 2 * cov @ w

    bounds = (
        [(0.0, None)] * N
        if long_only
        else [(None, None)] * N
    )   

    result = minimize(
        objective,
        initial_weights,
        jac=gradient,
        method="SLSQP",
        bounds=bounds,
        constraints={
            "type": "eq",
            "fun": lambda w: w.sum() - 1.0,
            "jac": lambda w: np.ones(N)
        },
        options={
            "ftol": tol,
            "maxiter": 1000,
            "disp": False
        }
    )

    if not result.success:
        raise RuntimeError(
            f"Optimización de mínima varianza fallida: {result.message}"
        )

    return result.x


# ------------------------------------------------------------
# Las seis estrategias calculadas desde los mismos momentos
# ------------------------------------------------------------

def build_strategies(mu, cov, gamma=GAMMA):
    """
    Construye 1/N, las cuatro variantes de Markowitz
    y mínima varianza a partir de mu y Sigma.
    """
    N = len(mu)
    #one = np.ones(N)

    # Benchmark 1/N
    w_equal_weight = np.repeat(1/N, N)

    # 1. Markowitz sin restricciones presupuestarias
    x_unconstrained = linear_solve(cov, mu) / gamma

    # 2. Solución anterior normalizada posteriormente
    net_exposure = x_unconstrained.sum()

    if np.isclose(net_exposure, 0.0, atol=1e-10):
        raise ValueError(
            "La solución sin restricciones tiene exposición neta casi cero; "
            "no se puede normalizar de forma estable."
        )

    w_normalized = x_unconstrained / net_exposure

    # 3. Markowitz con 1'w = 1 y cortos permitidos
    w_budget = solve_markowitz(
        mu,
        cov,
        gamma=gamma,
        long_only=False
    )

    # 4. Markowitz con 1'w = 1 y sin cortos
    w_long_only = solve_markowitz(
        mu,
        cov,
        gamma=gamma,
        long_only=True
    )

    # Mínima varianza long-only
    w_minimum_variance = solve_minimum_variance(
        cov,
        long_only=True
    )

    return {
        "benchmark 1/N": w_equal_weight,
        "sin restricciones": x_unconstrained,
        "normalizada posteriormente": w_normalized,
        "presupuesto, cortos permitidos": w_budget,
        "presupuesto + long-only": w_long_only,
        "mínima varianza + long-only": w_minimum_variance
    }


# ------------------------------------------------------------
# Motor único de backtest
# ------------------------------------------------------------

def run_backtest(
    returns,
    covariance_estimator,
    estimator_name,
    window=WINDOW,
    gamma=GAMMA
):
    """
    Ejecuta exactamente las mismas seis estrategias.

    La única diferencia entre ejecuciones será covariance_estimator:
    - sample_covariance
    - ledoit_wolf_covariance
    """
    portfolio_returns = {}
    weights_history = {}
    diagnostics = []

    for i in range(window, len(returns)):

        # Información disponible hasta el cierre del mes anterior.
        estimation_window = returns.iloc[i - window:i]

        # Retorno fuera de muestra del mes siguiente.
        test_return = returns.iloc[i]

        # La media es siempre muestral.
        mu = estimation_window.mean().to_numpy()

        # Sólo cambia el método de estimación de Sigma.
        cov, shrinkage = covariance_estimator(estimation_window)

        current_weights = build_strategies(
            mu,
            cov,
            gamma=gamma
        )

        if not portfolio_returns:
            portfolio_returns = {
                name: [] for name in current_weights
            }

            weights_history = {
                name: [] for name in current_weights
            }

        for name, w in current_weights.items():
            portfolio_return = w @ test_return.to_numpy()

            portfolio_returns[name].append(
                (test_return.name, portfolio_return)
            )

            weights_history[name].append(w)

        diagnostics.append({
            "date": test_return.name,
            "estimador": estimator_name,
            "shrinkage": shrinkage,
            "condition_number": np.linalg.cond(cov),
            "minimum_eigenvalue": np.linalg.eigvalsh(cov).min()
        })

    oos_returns = pd.DataFrame(
        {
            name: pd.Series(dict(values))
            for name, values in portfolio_returns.items()
        }
    )

    weights = {
        name: pd.DataFrame(
            values,
            index=oos_returns.index,
            columns=returns.columns
        )
        for name, values in weights_history.items()
    }

    diagnostics = pd.DataFrame(
        diagnostics
    ).set_index("date")

    return oos_returns, weights, diagnostics


# ============================================================
# 4. BACKTEST CON COVARIANZA MUESTRAL
# ============================================================

oos_sample, weights_sample, diagnostics_sample = run_backtest(
    returns=returns,
    covariance_estimator=sample_covariance,
    estimator_name="Muestral"
)


# ============================================================
# 5. BACKTEST CON COVARIANZA LEDOIT-WOLF
# ============================================================

oos_lw, weights_lw, diagnostics_lw = run_backtest(
    returns=returns,
    covariance_estimator=ledoit_wolf_covariance,
    estimator_name="Ledoit-Wolf"
)


# ============================================================
# 6. VALIDACIONES
# ============================================================

for weights_set in [weights_sample, weights_lw]:

    assert np.allclose(
        weights_set["benchmark 1/N"].sum(axis=1),
        1.0,
        atol=1e-6
    )

    assert np.allclose(
        weights_set["normalizada posteriormente"].sum(axis=1),
        1.0,
        atol=1e-6
    )

    assert np.allclose(
        weights_set["presupuesto, cortos permitidos"].sum(axis=1),
        1.0,
        atol=1e-6
    )

    assert np.allclose(
        weights_set["presupuesto + long-only"].sum(axis=1),
        1.0,
        atol=1e-6
    )

    assert np.allclose(
        weights_set["mínima varianza + long-only"].sum(axis=1),
        1.0,
        atol=1e-6
    )

    assert (
        weights_set["presupuesto + long-only"] >= -TOL
    ).all().all()

    assert (
        weights_set["mínima varianza + long-only"] >= -TOL
    ).all().all()


# ============================================================
# 7. MÉTRICAS DE RESULTADOS
# ============================================================

def performance_summary(portfolio_return, portfolio_weights):
    monthly_volatility = portfolio_return.std(ddof=1)

    return pd.Series({
        "rentabilidad media mensual": portfolio_return.mean(),
        "volatilidad mensual": monthly_volatility,
        "rentabilidad anualizada": (
            (1 + portfolio_return.mean()) ** 12 - 1
        ),
        "volatilidad anualizada": monthly_volatility * np.sqrt(12),
        "Sharpe anualizado (rf=0)": (
            portfolio_return.mean() / monthly_volatility * np.sqrt(12)
            if monthly_volatility > 0 else np.nan
        ),
        "rentabilidad acumulada": (
            (1 + portfolio_return).prod() - 1
        ),
        "peso máximo absoluto": (
            portfolio_weights.abs().to_numpy().max()
        ),
        "porcentaje de pesos negativos": (
            portfolio_weights < -TOL
        ).mean().mean(),
        "turnover medio simple": (
            portfolio_weights.diff().abs().sum(axis=1).iloc[1:].mean()
        )
    })


summary_sample = pd.DataFrame(
    {
        name: performance_summary(
            oos_sample[name],
            weights_sample[name]
        )
        for name in oos_sample.columns
    }
)

summary_lw = pd.DataFrame(
    {
        name: performance_summary(
            oos_lw[name],
            weights_lw[name]
        )
        for name in oos_lw.columns
    }
)


# ============================================================
# 8. COMPARACIÓN FINAL
# ============================================================

comparison_summary = pd.concat(
    {
        "Covarianza muestral": summary_sample,
        "Ledoit-Wolf": summary_lw
    },
    axis=1
)

print("COMPARACIÓN DE ESTRATEGIAS")
display(comparison_summary)

print("DIAGNÓSTICO: COVARIANZA MUESTRAL")
display(diagnostics_sample.describe())

print("DIAGNÓSTICO: LEDOIT-WOLF")
display(diagnostics_lw.describe())


# Pesos de la primera fecha fuera de muestra, para ambos métodos
first_weights_sample = pd.DataFrame(
    {
        name: weights_sample[name].iloc[0]
        for name in weights_sample
    }
)

first_weights_lw = pd.DataFrame(
    {
        name: weights_lw[name].iloc[0]
        for name in weights_lw
    }
)

print("PESOS INICIALES: COVARIANZA MUESTRAL")
display(first_weights_sample)

print("PESOS INICIALES: LEDOIT-WOLF")
display(first_weights_lw)

array(['XOM', 'MSFT', 'AAPL', 'JNJ', 'PG', 'IBM', 'T', 'JPM', 'GE', 'CVX',
       'BAC', 'GOOGL', 'PFE', 'WFC', 'CSCO', 'KO', 'HPQ', 'WMT', 'MRK',
       'ORCL', 'PEP', 'VZ', 'PM', 'GS', 'ABT', 'SLB', 'COP', 'MCD', 'OXY',
       'RTX', 'C', 'DIS', 'MMM', 'AMGN', 'UPS', 'HD', 'MDT', 'AXP',
       'CMCSA', 'CVS', 'AMZN', 'BMY', 'USB', 'CL', 'MO', 'MS', 'BA', 'V',
       'GILD', 'TGT'], dtype=object)

[                       0%                       ]

Número de activos del universo: 50


[*********************100%***********************]  50 of 50 completed


Observaciones mensuales: 179
Primera rentabilidad: 2010-02-28
Última rentabilidad: 2024-12-31
COMPARACIÓN DE ESTRATEGIAS


...

,shrinkage,condition_number,minimum_eigenvalue
count,119.000000,119.000000,119.000000
mean,0.163135,110.982338,0.000802
std,0.028908,35.296598,0.000245
min,0.112395,50.964517,0.000446
25%,0.133944,82.239491,0.000596
50%,0.166399,109.498082,0.000709
75%,0.188696,136.399681,0.001031
max,0.226721,176.909315,0.001147


PESOS INICIALES: COVARIANZA MUESTRAL


,benchmark 1/N,sin restricciones,normalizada posteriormente,"presupuesto, cortos permitidos",presupuesto + long-only,mínima varianza + long-only
Ticker,,,,,,
AAPL,0.02,63.455327,0.434384,27.293690,5.555023e-01,4.047002e-02
ABT,0.02,-147.342065,-1.008630,-118.886801,0.000000e+00,0.000000e+00
AMGN,0.02,-68.281658,-0.467422,-34.815522,6.030073e-17,2.201948e-02
AMZN,0.02,-30.848229,-0.211172,-6.212353,0.000000e+00,1.666843e-18
AXP,0.02,43.100893,0.295047,28.663085,0.000000e+00,2.350688e-18
BA,0.02,-16.305570,-0.111620,-48.898802,2.957603e-17,1.713870e-18
BAC,0.02,-38.524816,-0.263722,-60.533710,2.010258e-17,1.361425e-02
BMY,0.02,85.719050,0.586790,57.255791,0.000000e+00,5.316562e-02
C,0.02,-49.649668,-0.339877,-11.047327,0.000000e+00,4.175535e-19


PESOS INICIALES: LEDOIT-WOLF


,benchmark 1/N,sin restricciones,normalizada posteriormente,"presupuesto, cortos permitidos",presupuesto + long-only,mínima varianza + long-only
Ticker,,,,,,
AAPL,0.02,8.091789,0.212429,5.123154,5.574891e-01,4.156247e-02
ABT,0.02,-8.733939,-0.229287,-7.403627,0.000000e+00,0.000000e+00
AMGN,0.02,-0.967602,-0.025402,-1.273204,1.258871e-17,1.597083e-02
AMZN,0.02,-0.891166,-0.023395,-0.110275,5.932585e-17,0.000000e+00
AXP,0.02,3.915978,0.102804,4.116728,3.891316e-17,0.000000e+00
BA,0.02,-4.155906,-0.109103,-3.818555,0.000000e+00,0.000000e+00
BAC,0.02,-5.150828,-0.135222,-8.308398,3.209603e-17,1.386888e-02
BMY,0.02,9.912205,0.260219,6.003580,0.000000e+00,6.084483e-02
C,0.02,-4.598926,-0.120733,-0.918457,0.000000e+00,2.170436e-19
